In [ ]:
import numpy as np

class Perceptron:

  def __init__(self, init_weights, bias):
    #init_weights is initial weight vector
    self.weights = np.array(init_weights) # initialize weight vector
    self.bias = bias # init bias value
    self.delta = 0 # init delta value
    self.vector_length = len(self.weights)

    self.index = 0 # index value of perceptron in a layer
    self.error = 0 # only really useful for output nodes

  def get_activation(self):
    # selc.calc_activation(input_vector)
    return(self.activation)

  def set_index(self, index):
    self.index = index



  def calc_activity(self, inputs):
    # input vector and compute activity value, assign to activity field
    inputs = np.array(inputs)
    self.activity = np.dot(self.weights, inputs) + self.bias

  def calc_activation(self):
    # input activity value and compute activation function value, assign to activation field
    self.activation = 1 / (1.0 + np.exp(-1 * self.activity))

  def set_delta_weights(self, inputs, target, eta):
    # calculate delta_weights from input vector and learning rate
    inputs = np.array(inputs)
    error = target - self.activation
    derivative = (1 - self.activation) * self.activation
    delta = error * derivative
    self.delta_weights = eta * delta * inputs
    self.delta_bias = eta * delta

  def update_weights(self):
    # update weights using values of set_delta_weights fn
    self.weights += self.delta_weights
    self.bias += self.delta_bias

  def train(self, inputs, target, eta, epochs):
    # iteratively update weights and biases
    for epoch in range(epochs):
      self.calc_activity(inputs)
      self.calc_activation()
      self.set_delta_weights(inputs, target, eta)
      self.update_weights()
      if epoch > 7474:
        print(f"Epoch {epoch + 1} | Activation Value: {self.activation}, Weights: {self.weights}")


weights = [0.24, 0.88]
inputs = [0.8, 0.9]
target = 0.85
bias = 0
eta = 5.0

percy = Perceptron(weights, bias)

In [ ]:
percy.calc_activity(inputs)
percy.calc_activation()

print(percy.activation)

In [ ]:
percy.train(inputs, target, eta, 75)

In [ ]:
# calculating loss gradient
percy.activation = 0.3
error = target - percy.activation
derivative = (1 - percy.activation) * percy.activation
delta = -1 * error * derivative
print(delta)

In [20]:
# multi level perceptron

import numpy as np
import math

class Perceptron:
    def __init__(self, input_size, layer_sizes, initial_weights, initial_biases):
        self.layers = []
        self.layer_sizes = [input_size] + layer_sizes
        self.initial_weights = initial_weights
        self.initial_biases = initial_biases
        self.initialize_network()

    def calc_activity(self, weights, inputs, biases):
        return np.dot(weights, inputs) + biases


    def activation(self, A): # sigmoid activation function
        return 1 / (1 + np.exp(-A))

    def derivative(self, x): # sigmoid derivative
        return x * (1 - x)

    def calc_layers(self, input_size):
        self.layer_sizes.insert(0, input_size)


    def get_initial_values(self, layer):
        weights_shape = (self.layer_sizes[layer], self.layer_sizes[layer - 1])
        biases_shape = (self.layer_sizes[layer],)

        # print(f"Layer {layer} (shape {weights_shape[0]}x{weights_shape[1]} for weights, {biases_shape[0]} for biases):")

        weights = np.zeros(weights_shape)
        for i in range(weights_shape[0]):
            for j in range(weights_shape[1]):
                weights[i, j] = float(input(f"Layer {layer} | Enter weight [{i}][{j}]: "))

        biases = np.zeros(biases_shape)
        for i in range(biases_shape[0]):
            biases[i] = float(input(f"Layer {layer} | Enter bias [{i}]: "))

        return weights, biases

    # def initialize_network(self):
    #     for layer_index in range(1, len(self.layer_sizes)):
    #         weights, biases = self.get_initial_values(layer_index)
    #         self.layers.append({'weights': weights, 'biases': biases, 'activations': None, 'deltas': None})


    def initialize_network(self):
        for i, layer_size in enumerate(self.layer_sizes[1:], 1):  # Skip input layer
            weights = np.array(self.initial_weights[i-1])
            biases = np.array(self.initial_biases[i-1])
            self.layers.append({'weights': weights, 'biases': biases, 'activations': None, 'deltas': None})

    # def feed_forward(self, inputs):
    #     activations = np.array(inputs)
    #     for layer_index, layer in self.layers:
    #         activities = self.calc_activity(layer['weights'], activations, layer['biases'])
    #         activations = self.activation(activities)
    #         layer['activations'] = activations

    #         print(f"Layer {layer_index + 1} activation values:")
    #         for node_index, activation_value in enumerate(activations):
    #             print(f"Node {node_index + 1}: {activation_value}")
    #     return activations

    def feed_forward(self, inputs):
      activations = np.array(inputs)
      for layer_index, layer in enumerate(self.layers):  # Correct usage of enumerate
          activities = self.calc_activity(layer['weights'], activations, layer['biases'])
          activations = self.activation(activities)
          layer['activations'] = activations

          # Print activation values in a more compact format
          activation_values = ", ".join(f"{activation:.4f}" for activation in activations)
          print(f"Layer {layer_index + 1} activations: [{activation_values}]")
      return activations


    # def backprop(self, inputs, target, eta):
    #     for i in reversed(range(len(self.layers))):
    #         layer = self.layers[i]
    #         if i == len(self.layers) - 1:
    #             error = np.array(target) - layer['activations']
    #             deltas = error * self.derivative(layer['activations'])
    #         else:
    #             next_layer = self.layers[i + 1]
    #             error = np.dot(next_layer['weights'].T, next_layer['deltas'])
    #             layer['deltas'] = error * self.derivative(layer['activations'])

    def backprop(self, inputs, target, eta):
      # Calculate deltas for the output layer
      output_layer = self.layers[-1]
      error = target - output_layer['activations']
      output_layer['deltas'] = error * self.derivative(output_layer['activations'])

      # Backpropagate deltas to hidden layers
      for i in reversed(range(len(self.layers) - 1)):  # Exclude the output layer
          layer = self.layers[i]
          next_layer = self.layers[i + 1]
          # Here, you're using the 'deltas' from the next layer, which is correctly initialized by this point
          error = np.dot(next_layer['weights'].T, next_layer['deltas'])
          layer['deltas'] = error * self.derivative(layer['activations'])



    def update_weights(self, inputs, eta):
      for i, layer in enumerate(self.layers):
          if i == 0:
              inputs_to_use = np.array(inputs)
          else:
              inputs_to_use = self.layers[i - 1]['activations']
          layer['weights'] += eta * np.outer(layer['deltas'], inputs_to_use)
          layer['biases'] += eta * layer['deltas']

          # Print updated weights and biases in a more structured format
          weights_str = np.array2string(layer['weights'], formatter={'float_kind':'{0:.4f}'.format})
          biases_str = np.array2string(layer['biases'], formatter={'float_kind':'{0:.4f}'.format})
          print(f"Layer {i + 1} | Updated weights and biases:\nWeights:\n{weights_str}\nBiases:\n{biases_str}")


    def train(self, inputs, d, eta, epochs):

        print_interval = max(1, math.floor(epochs / 10))

        for epoch in range(epochs):
            self.feed_forward(inputs)
            self.backprop(inputs, d, eta)
            self.update_weights(inputs, eta)
            if epoch % print_interval == 0 or epoch == epochs - 1:
                output = self.feed_forward(inputs)
                loss = np.mean((np.array(d) - output) ** 2)
                print(f"Epoch {epoch + 1} | Loss: {loss}")




In [21]:
input_size = 2
layer_sizes = [2, 1]

initial_weights = [
    np.array([[0.8, 0.5], [0.1, 0.2]]),
    np.array([[0.2, 0.7]])
]

initial_biases = [
    np.array([0.35, 0.35]),
    np.array([0.60])
]

mlp = Perceptron(input_size, layer_sizes, initial_weights, initial_biases)


In [22]:
inputs = np.array([1, 3])
d = np.array([0.95])
eta = 0.1
epochs = 100
# mlp.layers = [(layer['weights'], layer['biases']) for layer in mlp.layers]

mlp.train(inputs, d, eta, epochs)

Layer 1 activations: [0.9340, 0.7408]
Layer 2 activations: [0.7867]
Layer 1 | Updated weights and biases:
Weights:
[[0.8000 0.5001]
 [0.1004 0.2011]]
Biases:
[0.3500 0.3504]
Layer 2 | Updated weights and biases:
Weights:
[[0.2026 0.7020]]
Biases:
[0.6027]
Layer 1 activations: [0.9340, 0.7416]
Layer 2 activations: [0.7879]
Epoch 1 | Loss: 0.026264829772433995
Layer 1 activations: [0.9340, 0.7416]
Layer 2 activations: [0.7879]
Layer 1 | Updated weights and biases:
Weights:
[[0.8001 0.5002]
 [0.1007 0.2022]]
Biases:
[0.3501 0.3507]
Layer 2 | Updated weights and biases:
Weights:
[[0.2051 0.7040]]
Biases:
[0.6054]
Layer 1 activations: [0.9341, 0.7423]
Layer 2 activations: [0.7891]
Layer 1 | Updated weights and biases:
Weights:
[[0.8001 0.5003]
 [0.1011 0.2033]]
Biases:
[0.3501 0.3511]
Layer 2 | Updated weights and biases:
Weights:
[[0.2076 0.7060]]
Biases:
[0.6081]
Layer 1 activations: [0.9341, 0.7431]
Layer 2 activations: [0.7903]
Layer 1 | Updated weights and biases:
Weights:
[[0.8001 0.5

In [5]:

mlp.layers = [(layer['weights'], layer['biases']) for layer in mlp.layers]
print(mlp.layers)

[{'weights': array([[0.8, 0.5],
       [0.1, 0.2]]), 'biases': array([0., 0.]), 'activations': None, 'deltas': None}, {'weights': array([[0.2, 0.7]]), 'biases': array([0.]), 'activations': None, 'deltas': None}]
